In [1]:
from model.lightning_model import GPTLightningModule
import torch
from tokenizers import Tokenizer

path_riddles = "Checkpoints/"

Importing model from saved checkpoint

In [2]:
# Carica dal file salvato
tokenizer = Tokenizer.from_file(path_riddles + "custom_tokenizer_tinystories.json")

In [3]:
# 1. Initialize your model architecture
model = GPTLightningModule(
    block_size=128,
    vocab_size=4000,
    n_embd=256,
    n_head=8,
    n_layer=6,
    use_qat=False,  # Default
    learning_rate = 1e-3,
    dropout = 0.1,
    weight_decay=1e-1
)

# 2. Load the checkpoint file using standard torch
import torch
checkpoint = torch.load(path_riddles + "R-gpt-val_loss=2.76.ckpt", map_location="cpu")

# 3. Load the weights into the model
# Note: If 'checkpoint' is just the weights, use it directly. 
# If it's a dict with a 'state_dict' key, use checkpoint['state_dict']
if "state_dict" in checkpoint:
    model.load_state_dict(checkpoint["state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

GPTLightningModule(
  (model): GPTModel(
    (token_embedding_table): Embedding(4000, 256)
    (position_embedding_table): Embedding(128, 256)
    (emb_dropout): Dropout(p=0.1, inplace=False)
    (blocks): Sequential(
      (0): Block(
        (mha): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (ffwd): FeedFoward(
          (net): Sequential(
            (0): Linear(in_features=256, out_features=1024, bias=True)
            (1): ReLU()
            (2): Linear(in_features=1024, out_features=256, bias=True)
            (3): Dropout(p=0.1, inplace=False)
          )
        )
        (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
      (1): Block(
        (mha): MultiheadAttention(
          (out_proj): NonDyn

In [4]:
#model = GPTLightningModule.load_from_checkpoint(path_riddles +"R-gpt-val_loss=2.76.ckpt")
model.to("cpu")
encoded = tokenizer.encode("[BOS] Tell me a riddle about a dog \n RIDDLE:")
input_ids = torch.tensor(encoded.ids)
input_ids = input_ids.reshape(1, -1)

In [5]:
from torch.profiler import profile, ProfilerActivity

model.eval()
with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    with torch.no_grad():
      model.generate(input_ids, 1, temperature=0.7, top_k=100)

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=15))

--------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                  Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
--------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
    aten::_native_multi_head_attention         6.67%       1.268ms        35.16%       6.686ms       1.114ms             6  
                          aten::linear         2.24%     425.600us        25.84%       4.913ms     377.923us            13  
                           aten::addmm        21.27%       4.044ms        24.33%       4.627ms     243.526us            19  
                      aten::layer_norm         0.90%     171.200us         9.47%       1.800ms     138.485us            13  
                              aten::mm         9.01%       1.713ms         9.08%       1.726ms     287.633us             6  


c:\Users\Giovanni\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\profiler\profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [6]:
def score_attention_heads(lightning_module, dataloader, num_batches=10):
    model = lightning_module.model
    n_head = model.blocks[0].mha.num_heads
    n_layer = len(model.blocks)
    head_dim = model.blocks[0].mha.embed_dim // n_head

    importance = [[0.0] * n_head for _ in range(n_layer)]
    hooks = []

    def make_hook(layer_idx):
        def hook(module, input, output):
            # output is a tuple: (attn_output, attn_weights)
            # attn_output shape: (B, T, n_embd)
            attn_out = output[0].detach()
            B, T, E = attn_out.shape
            per_head = attn_out.view(B, T, n_head, head_dim)  # (B, T, H, head_dim)
            scores = per_head.norm(dim=-1).mean(dim=(0, 1))    # (H,)
            for h in range(n_head):
                importance[layer_idx][h] += scores[h].item()
        return hook

    # ✅ correct attribute name: .mha not .attn
    for i, block in enumerate(model.blocks):
        hooks.append(block.mha.register_forward_hook(make_hook(i)))

    lightning_module.eval()
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if batch_idx >= num_batches:
                break
            input_ids = batch['x'].to("cpu")
            lightning_module(input_ids)

    for h in hooks:
        h.remove()

    for layer_idx in range(n_layer):
        for h in range(n_head):
            importance[layer_idx][h] /= num_batches

    return importance

In [12]:
import torch.nn as nn
def structured_prune_ffn(block, amount=0.3):
    """
    Physically shrinks FeedFoward: Linear(n_embd, 4*n_embd) -> Linear(n_embd, k)
    by removing the weakest output neurons of net[0] (and matching rows in net[2]).
    """
    fc1 = block.ffwd.net[0]  # (4*n_embd, n_embd)
    fc2 = block.ffwd.net[2]  # (n_embd, 4*n_embd)

    hidden_size = fc1.out_features
    num_to_keep = int(hidden_size * (1 - amount))

    # Score neurons by L2 norm of fc1 output rows
    scores = fc1.weight.data.norm(dim=1)  # (4*n_embd,)
    keep_indices = scores.topk(num_to_keep).indices.sort().values

    # Build new smaller layers
    new_fc1 = nn.Linear(fc1.in_features, num_to_keep, bias=fc1.bias is not None)
    new_fc2 = nn.Linear(num_to_keep, fc2.out_features, bias=fc2.bias is not None)

    # Copy surviving weights
    new_fc1.weight.data = fc1.weight.data[keep_indices, :]
    new_fc2.weight.data = fc2.weight.data[:, keep_indices]
    if fc1.bias is not None:
        new_fc1.bias.data = fc1.bias.data[keep_indices]
    if fc2.bias is not None:
        new_fc2.bias.data = fc2.bias.data  # fc2 bias is unaffected

    # Replace in-place inside the Sequential
    block.ffwd.net[0] = new_fc1
    block.ffwd.net[2] = new_fc2

    return num_to_keep

In [8]:
def structured_prune_attention_safe(block, heads_to_remove: list[int]):
    """
    Removes heads by slicing out_proj columns to zero — then rebuilds
    a smaller out_proj that simply skips removed heads' V computation.
    The residual stream dimension stays intact.
    """
    mha = block.mha
    n_embd   = mha.embed_dim
    n_head   = mha.num_heads
    head_dim = n_embd // n_head

    keep_heads = sorted(set(range(n_head)) - set(heads_to_remove))

    # Only prune V and out_proj — Q/K are left so attention scores still work
    v_rows_remove = [
        r for h in heads_to_remove
        for r in range(2 * n_embd + h * head_dim,
                       2 * n_embd + (h + 1) * head_dim)
    ]
    out_cols_remove = [
        c for h in heads_to_remove
        for c in range(h * head_dim, (h + 1) * head_dim)
    ]

    with torch.no_grad():
        mha.in_proj_weight.data[v_rows_remove, :] = 0.0
        mha.out_proj.weight.data[:, out_cols_remove] = 0.0
        if mha.in_proj_bias is not None:
            mha.in_proj_bias.data[v_rows_remove] = 0.0

In [9]:
train_dataset, val_dataset, test_dataset = torch.load("dataset.pt", weights_only= "True")

from torch.utils.data import DataLoader
from utils.data import SLM_dataset
bs = 32
cs = 128

train = SLM_dataset(train_dataset, cs)
val = SLM_dataset(val_dataset, cs)

train_dataloader = DataLoader(train, batch_size=bs, shuffle=True, num_workers=0)
val_dataloader = DataLoader(val, batch_size=bs, shuffle=False, num_workers=0)

c:\Users\Giovanni\Documents\Projects\Small_LMs\utils\data.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  masks.append(torch.tensor(padding_mask, dtype=torch.bool))


In [13]:
# Score heads first (reuse function from before)
importance = score_attention_heads(model, train_dataloader, num_batches=10)

# Decide which heads to remove per layer (bottom 30%)
all_heads = [
    (l, h, importance[l][h])
    for l in range(len(model.model.blocks))
    for h in range(len(importance[l]))
]
all_heads.sort(key=lambda x: x[2])
num_to_prune = int(len(all_heads) * 0.3)

# Group by layer
from collections import defaultdict
heads_per_layer = defaultdict(list)
for l, h, _ in all_heads[:num_to_prune]:
    heads_per_layer[l].append(h)

# Apply structured pruning
for layer_idx, block in enumerate(model.model.blocks):
    # FFN — physically smaller matrices
    new_hidden = structured_prune_ffn(block, amount=0.3)
    print(f"Block {layer_idx} FFN: 4*n_embd → {new_hidden} neurons")

    # Attention — safe head removal
    if layer_idx in heads_per_layer:
        structured_prune_attention_safe(block, heads_per_layer[layer_idx])
        print(f"Block {layer_idx} Attention: removed heads {heads_per_layer[layer_idx]}")

Block 0 FFN: 4*n_embd → 716 neurons
Block 0 Attention: removed heads [6, 1, 5, 7, 2, 4, 0]
Block 1 FFN: 4*n_embd → 716 neurons
Block 1 Attention: removed heads [5, 6, 1, 4, 7, 0]
Block 2 FFN: 4*n_embd → 716 neurons
Block 2 Attention: removed heads [6]
Block 3 FFN: 4*n_embd → 716 neurons
Block 4 FFN: 4*n_embd → 716 neurons
Block 5 FFN: 4*n_embd → 716 neurons


In [ ]:
# Save the pruned architecture state + any metadata you need
checkpoint = {
    # Model weights
    "state_dict": model.state_dict(),
    
    # Original hparams (for reference)
    "original_hparams": dict(model.hparams),
    
    # Pruning metadata — needed to reconstruct the actual current architecture
    "pruning_metadata": {
        "ffn_hidden_sizes": [
            block.ffwd.net[0].out_features      # actual post-pruning size
            for block in model.model.blocks
        ],
        "attention_heads_remaining": [
            block.mha.num_heads
            for block in model.model.blocks
        ],
        "pruning_amount_ffn": 0.3,
        "pruning_percent_heads": 0.3,
    }
}

torch.save(checkpoint, "pruned_model_finetunable.pt")
print("Saved ✅")

In [21]:
model.hparams.learning_rate = 1e-4
model.hparams.dropout = 0.2

In [24]:
from model.train import run_training
model.train()

best_model = run_training(
        train_loader = train_dataloader,
        val_loader = val_dataloader,
        save_dir = path_riddles,
        experiment_name="pruned_riddles", # Dai nomi significativi!
        model = model,
        max_epochs = 2,
        patience=3 # Ferma se non migliora per 8 epoche
    )

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


--- Starting Experiment: pruned_riddles ---


┏━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ GPTModel │  5.9 M │ train │     0 │
└───┴───────┴──────────┴────────┴───────┴───────┘

Trainable params: 5.9 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 5.9 M                                                                                                
Total estimated model params size (MB): 23                                                                         
Modules in train mode: 85                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Metric val_loss improved. New best score: 2.877
Metric val_loss improved by 0.010 >= min_delta = 0.0. New best score: 2.867
`Trainer.fit` stopped: `max_epochs=2` reached.


Training completato. Il miglior modello è salvato in:
C:\Users\Giovanni\Documents\Projects\Small_LMs\Checkpoints\pruned_riddles\gpt-epoch=01-val_loss=2.8666.ckpt
